# Retail Sales & Customer Analytics
A reproducible SQL + pandas walkthrough. **The bundled dataset is synthetic demo data. Do not present its numbers as real business evidence.**

## 1. Load synthetic demo (replace with real UCI workbook after downloading)
Run from project root or change `ROOT` accordingly.

In [ ]:
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
df = pd.read_csv(ROOT / "data/demo/online_retail_demo.csv")
print(df.shape)
df.head()

## 2. Run the shared pipeline, avoiding conflicting cleaning rules

In [ ]:
import sys
sys.path.insert(0, str(ROOT / "scripts"))
from run_pipeline import transform, outputs
result = transform(df)
summary = outputs(result, "demo")
summary

## 3. Track business KPIs: gross merchandise revenue and order value

In [ ]:
monthly = result["monthly_sales"]
monthly[["order_month", "gross_sales", "orders", "aov"]].head(12)

## 4. Segment customers by RFM
Scores are derived from the dataset itself; they do not prove campaign effectiveness.

In [ ]:
rfm = result["rfm"]
rfm.groupby("segment").agg(customers=("customer_id", "count"), spend=("monetary", "sum")).sort_values("spend", ascending=False)

## 5. Run SQL CTE/window-function analyses in SQLite

In [ ]:
import sqlite3
with sqlite3.connect(ROOT / "database/retail_analysis.sqlite") as con:
    sql = (ROOT / "sql/06_customer_rank.sql").read_text(encoding="utf-8")
    display(pd.read_sql_query(sql, con).head(10))

## 6. For real data
Download the original UCI Online Retail workbook (source cited in README), then run `python scripts/run_pipeline.py --mode real`. Review the data quality assumptions and the last partial month before drawing business conclusions.